In [ ]:
import numpy as np
import pandas as pd

# 1. 讀取 2026 年多站原始檔案
input_file = 'taoyuan_sites_air_quality_2026_01_07.csv'
df_raw = pd.read_csv(input_file)

# 2. Melt 解除旋轉 (將 monitorvalue00 ~ monitorvalue23 轉為縱列)
hours_cols = [f'monitorvalue{i:02d}' for i in range(24)]

df_melted = df_raw.melt(
    id_vars=['sitename', 'monitordate', 'itemengname'],
    value_vars=hours_cols,
    var_name='Hour_Col',
    value_name='Value',
)

# 組合時間格式 YYYY/MM/DD HH:00:00
df_melted['Hour'] = df_melted['Hour_Col'].str.replace('monitorvalue', '')
df_melted['Date_Only'] = df_melted['monitordate'].str.replace('-', '/')
df_melted['publishtime'] = (
    df_melted['Date_Only'] + ' ' + df_melted['Hour'] + ':00:00'
)

# 3. 測項英文名稱對應表
measure_map = {
    'PM2.5': 'pm2.5',
    'PM10': 'pm10',
    'O3': 'o3',
    'CO': 'co',
    'SO2': 'so2',
    'NO': 'no',
    'NO2': 'no2',
    'NOx': 'nox',
    'WIND_SPEED': 'wind_speed',
    'WIND_DIREC': 'wind_direc',
    'AMB_TEMP': 'amb_temp',
    'RH': 'rh',
       }

# 篩選並轉換測項
df_filtered = df_melted[df_melted['itemengname'].isin(measure_map.keys())].copy()
df_filtered['測項_mapped'] = df_filtered['itemengname'].map(measure_map)

# Pivot 轉置為寬表格
df_pivoted = df_filtered.pivot_table(
    index=['sitename', 'publishtime'],
    columns='測項_mapped',
    values='Value',
    aggfunc='first',
).reset_index()

# 強制轉型為 float (確保非數值與無效值轉為 NaN)
cols_to_num = [
    'no',
    'no2',
    'nox',
    'o3',
    'pm2.5',
    'pm10',
    'co',
    'so2',        
    'amb_temp',
    'rh',
]
for c in cols_to_num:
    if c in df_pivoted.columns:
        df_pivoted[c] = pd.to_numeric(df_pivoted[c], errors='coerce')
    else:
        df_pivoted[c] = np.nan

# 4. 計算 7 項科學指標
# (1) 總氧化劑 Ox
df_pivoted['ox'] = np.round(df_pivoted['o3'] + df_pivoted['no2'], 3)

# (2) PM2.5 / PM10 比值
df_pivoted['pm_ratio'] = np.where(
    df_pivoted['pm10'] > 0,
    np.round(df_pivoted['pm2.5'] / df_pivoted['pm10'], 4),
    np.nan,
)

# (3) NMHC / THC 比值
#df_pivoted['nmhc_ratio'] = np.where(
 #   df_pivoted['thc'] > 0,
  #  np.round(df_pivoted['nmhc'] / df_pivoted['thc'], 4),
   # np.nan,)

# (4) 特徵氣體比值
df_pivoted['co_nox_ratio'] = np.where(
    df_pivoted['nox'] > 0,
    np.round(df_pivoted['co'] / df_pivoted['nox'], 4),
    np.nan,
)

df_pivoted['so2_nox_ratio'] = np.where(
    df_pivoted['nox'] > 0,
    np.round(df_pivoted['so2'] / df_pivoted['nox'], 4),
    np.nan,
)

# (5) 飽和水蒸氣壓 es 與 實際水蒸氣壓 e (hPa)
T = df_pivoted['amb_temp']
RH = df_pivoted['rh']
es = 6.112 * np.exp((17.67 * T) / (T + 243.5))
e = es * (RH / 100.0)

df_pivoted['sat_vapor_press'] = np.round(es, 3)
df_pivoted['vapor_press'] = np.round(e, 3)

# 5. 指定留下的 21 個目標欄位
specified_cols = [
    'sitename',
    'publishtime',
    'pm2.5',
    'pm10',
    'o3',
    'co',
    'so2',
    'no',
    'no2',
    'nox',
    'wind_speed',
    'wind_direc',
    'amb_temp',
    'rh',
    'ox',
    'pm_ratio',
    'co_nox_ratio',
    'so2_nox_ratio',
    'sat_vapor_press',
    'vapor_press',
]

# 補齊缺少欄位並過濾
for col in specified_cols:
    if col not in df_pivoted.columns:
        df_pivoted[col] = np.nan

df_2026_final = df_pivoted[specified_cols].copy()

# 6. 依時間與測站排序
df_2026_final.sort_values(by=['publishtime', 'sitename'], inplace=True)
df_2026_final.reset_index(drop=True, inplace=True)

# 7. 匯出 CSV (utf-8-sig 防止 Excel 中文亂碼)
output_filename = '2026_01_07桃園五站空品監測資料_精簡21欄合體版.csv'
df_2026_final.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f'處理完成！檔案已儲存為：{output_filename}')
print(f'資料總筆數：{len(df_2026_final)} 行')
print(
    f'欄位清單 ({len(df_2026_final.columns)}個)：{df_2026_final.columns.tolist()}'
)

In [ ]:
# https://data.moenv.gov.tw/dataset/detail/AQX_P_13
# https://drive.google.com/drive/u/2/folders/1b-Yc8dLtINesuGnbyCvLOjzpH-R-7ubT